<a href="https://colab.research.google.com/github/rrushil/224B/blob/main/224B_P1f2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import os
from tensorflow import keras
from tensorflow.keras import layers, models
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
from PIL import Image
import pandas as pd
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install kaggle

In [4]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rushilravindran","key":"9b7c3c751df731d0ef9f09e61f9bd52d"}'}

In [6]:
! mv kaggle.json ~/.kaggle/
! mkdir -p ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json
#importing the dataset
! kaggle competitions download -c bioengr-224-b-spring-25-course-project-1

In [7]:
#unzipping the dataset
!unzip bioengr-224-b-spring-25-course-project-1.zip

Archive:  bioengr-224-b-spring-25-course-project-1.zip
  inflating: testImages/testImages/10307.jpg  
  inflating: testImages/testImages/10338.jpg  
  inflating: testImages/testImages/10369.jpg  
  inflating: testImages/testImages/10479.jpg  
  inflating: testImages/testImages/10490.jpg  
  inflating: testImages/testImages/10693.jpg  
  inflating: testImages/testImages/10945.jpg  
  inflating: testImages/testImages/11196.jpg  
  inflating: testImages/testImages/11238.jpg  
  inflating: testImages/testImages/11375.jpg  
  inflating: testImages/testImages/11376.jpg  
  inflating: testImages/testImages/1166.jpg  
  inflating: testImages/testImages/12024.jpg  
  inflating: testImages/testImages/1205.jpg  
  inflating: testImages/testImages/12077.jpg  
  inflating: testImages/testImages/1213.jpg  
  inflating: testImages/testImages/12312.jpg  
  inflating: testImages/testImages/1232.jpg  
  inflating: testImages/testImages/12381.jpg  
  inflating: testImages/testImages/1243.jpg  
  inflatin

In [8]:
#setting up all the paths for the respective folders that are unzipped
test = '/content/testImages/testImages'
train_im = '/content/trainImages/trainImages'
train_m = '/content/trainMasks/trainMasks'

In [9]:
#Sorts and gets list of all files from the training images folder
file = sorted(os.listdir(train_im))
#Build image paths
train_image = [os.path.join(train_im, f) for f in file]
#making sure the masks correspond and looping through and having them as _mask for their paths
train_mask = [
    os.path.join(train_m, f.replace('.jpg', '_mask.png'))
    for f in file
]
#looping through the test images and setting up a path
test_image = sorted([
    os.path.join(test, f)
    for f in os.listdir(test)
    if f.endswith('.jpg')
])

In [10]:
# creating a function for the u-net to call loading the image and corresponding mask and image to train on
def load_image_mask(im_path, m_path):
    image = tf.io.read_file(im_path)
    image = tf.image.decode_jpeg(image, channels=1) #setting up a decode for the correct image type of .jpeg for the masks
    image = tf.cast(image, tf.float32) / 255.0

    mask = tf.io.read_file(m_path)
    mask = tf.image.decode_png(mask, channels=1) #setting up a decode for the correct image type of .png for the masks
    mask = tf.cast(mask, tf.float32) / 255.0
    return image, mask

In [11]:
# creating a function for the u-net to call for the test images specifically
def load_image(im_path):
    image = tf.io.read_file(im_path)
    image = tf.image.decode_jpeg(image, channels=1) #setting up a decode for the correct image type of .jpeg for the masks
    image = tf.cast(image, tf.float32) / 255.0
    return image

In [12]:
BATCH_SIZE = 8 #using batch size of 8 to hopefully not run out of memory since had isssues with this before
BUFFER_SIZE = 1000

#creating the training dataset loading the corresponding masks and images together for the model to train on
train_dataset = tf.data.Dataset.from_tensor_slices((train_image, train_mask))
train_dataset = train_dataset.map(load_image_mask, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=100)
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
#creating the test dataset loading the test images and calling the previous function to use the load_image function
test_dataset = tf.data.Dataset.from_tensor_slices(test_image)
test_dataset = test_dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

**Creating the Model**

In [13]:
def double_convolution(input, n_filters):
   #running two conv2D layers and ensuring no padding and outputting 512x512x64
   y = layers.Conv2D(n_filters, kernel_size = (3,3), padding = "same", activation = "relu", kernel_initializer = "he_normal")(input)
   y = layers.Conv2D(n_filters, kernel_size = (3,3), padding = "same", activation = "relu", kernel_initializer = "he_normal")(input)
   return y

In [14]:
def encoder(input, n_filters):
   l = double_convolution(input, n_filters) #calling the previous function to apply the two conv2D layers
   d = layers.MaxPool2D(2)(l) #downsampling
   d = layers.Dropout(0.3)(d) #preventing overfitting by applying dropout
   return l, d

In [21]:
def decoder(x, conv_features, n_filters):
   x = layers.Conv2DTranspose(n_filters, 3, 2, padding="same")(x) #deconvolution
   x = layers.concatenate([x, conv_features]) #concatenate the upsampled feature map
   x = layers.Dropout(0.3)(x) #preventing overfitting by applying dropout
   x = double_convolution(x, n_filters) #calling the two conv2D laters again
   return x

In [22]:
def build_unet_model():
    #taking the image size and shape
    inputs = layers.Input(shape=(512, 512, 1)) # for the image shape
    #constructing the encoder blocks
    s1, e1 = encoder(inputs, 64)
    s2, e2 = encoder(e1, 128)
    s3, e3 = encoder(e2, 256)
    s4, e4 = encoder(e3, 512)
    #next block
    block = double_convolution(e4, 1024)
    #constructing the decoder blocks
    d1 = decoder(block, s4, 512)
    d2 = decoder(d1, s3, 256)
    d3 = decoder(d2, s2, 128)
    d4 = decoder(d3, s1, 64)
    #Output layer
    outputs = layers.Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)
    #Model
    model = models.Model(inputs, outputs, name="U-Net")
    return model

In [23]:
unet = build_unet_model() #building the model

In [24]:
#compiling the u-net model and using a binary crossentropy with accuracy and loss for the black and white image
unet.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [25]:
NUM_EPOCHS = 20 #using 20 epochs to train
#fitting the dataset using the previous train dataset with the images and masks
history = unet.fit(
    train_dataset,
    epochs=NUM_EPOCHS,
)

Epoch 1/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 64s 313ms/step - accuracy: 0.9861 - loss: 0.0827
Epoch 2/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.9982 - loss: 0.0145
Epoch 3/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9981 - loss: 0.0132
Epoch 4/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9982 - loss: 0.0125
Epoch 5/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9981 - loss: 0.0114
Epoch 6/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9981 - loss: 0.0112
Epoch 7/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9982 - loss: 0.0119
Epoch 8/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9981 - loss: 0.0100
Epoch 9/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9981 - loss: 0.0098
Epoch 10/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9981 - loss: 0.0090
Epoch 11/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9982 - loss: 0.0088
Epoch 12/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/ste

**Saving the Test Masks**

In [26]:
def save_mask(pred_mask, filename, save_dir):
    #threshold the masks
    pred_mask = tf.squeeze(pred_mask, axis=-1) #ensuring image is 512x512
    pred_mask = tf.where(pred_mask > 0.5, 1, 0) #ensuring the masks are binary
    pred_mask = tf.cast(pred_mask * 255, tf.uint8).numpy()
    #saving the masks
    save_path = os.path.join(save_dir, filename)
    Image.fromarray(pred_mask).save(save_path)

In [27]:
#setting up an output directory for the predictive masks
output_dir = '/content/predictions'
os.makedirs(output_dir, exist_ok=True)
test_filenames = [os.path.basename(p) for p in test_image]

#making predictions using multiple for loops to predict the masks
for batch_images, batch_filenames in zip(test_dataset, tf.data.Dataset.from_tensor_slices(test_filenames).batch(BATCH_SIZE)):
    predictions = unet.predict(batch_images) #predicting the masks based on test images
    #replacing the .jpg to _mask.png to ensure that the processImages function can read this and process for submission.csv
    for pred_mask, fname in zip(predictions, batch_filenames.numpy()):
        fname_str = fname.decode('utf-8').replace('.jpg', '_mask.png')
        save_mask(pred_mask, fname_str, output_dir) #saving the mask

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step


In [28]:
def processImages(imgDirectory: str, saveDirectory: str = os.getcwd(), returnDF:bool = False) -> pd.DataFrame | None:
    """
    Process binarized predicted mask images saved in a directory, implement checks,
    and create a `'submission.csv'` submission file containing image status and mask indices.

    **Do NOT modify this function.**

    Parameters
    ----------
    imgDirectory : str
        The directory containing the images to be processed. It should have exactly 127 .png files.
        When you save your model's predicted masks, make sure the pixel values
        are either 0 or 255, and save it as a .png file (preferably using PIL).
    saveDirectory : str, optional
        The directory where the resulting DataFrame will be saved as a CSV file.
        Defaults to the current directory.
    returnDF : bool, optional
        Whether to return the DataFrame. Defaults to False.

    Returns
    -------
    df: A DataFrame with columns 'imageID', 'status', and 'mask', indexed by 'imageID' if `returnDF` is True, else None

    Raises
    ------
    ValueError
        If the number of .png files in `imgDirectory` is not 127,
        if any image is not binary, or if any image is not 512x512 pixels.

    Example Usage
    -------------
    `processImages('path/to/img/folder', 'path/to/save/folder')`
    """

    files = [f for f in os.listdir(imgDirectory) if f.endswith('.png')]  # Get all .png files in the directory
    if len(files) != 127:
        raise ValueError("Directory must contain exactly 127 .png files")

    files.sort(key=lambda x: int(x.split('_')[0]))  # Sort the files

    data = []  # List of dictionaries to be converted to DataFrame
    for file in files:
        imgPath = os.path.join(imgDirectory, file)
        img = np.array(Image.open(imgPath).convert('L'), dtype=np.uint8)

        # Check if image is binary
        if not np.array_equal(img, img.astype(bool).astype(img.dtype) * 255):
            raise ValueError(f"Image {file} is not binary")
        # Check image size
        if img.shape != (512, 512):
            raise ValueError(f"Image {file} is not of size 512x512")

        status = 1 if np.any(img == 255) else 0  # Determine status of image
        maskIndices = ' '.join(map(str, np.nonzero(img.flatten() == 255)[0])) if status else '-100'

        data.append({'imageID': int(file.split('_')[0]), 'status': status, 'mask': maskIndices})

    df = pd.DataFrame(data).set_index('imageID')
    df.to_csv(os.path.join(saveDirectory, 'submission.csv'))

    if returnDF: return df

In [ ]:
#setting up a save directory and calling process images to export submission.csv
save_dir = '/content/save'
processImages(output_dir,save_dir)